# Poster Figures (v4) — Feature Columns + Sharpe Bar Chart + Best Strategy Chart

What this notebook gives you (screenshot/export ready):
1) Feature group map (group → specific feature columns)
2) Feature set map (F1–F5 expanded to specific feature columns)
3) Sharpe bar chart by feature set with (model|horizon) bars (highlights Logistic|3D)
4) Best-performance chart: shows the actual equity curve image for `F5 + Logistic + 3D` (if it exists)

Prereq (run once before this notebook):
```bash
python -m src.pipeline --mode live --notebook 02
```


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
ROOT = HERE if (HERE / 'outputs').exists() else HERE.parent
OUT_METRICS = ROOT / 'outputs' / 'metrics'
OUT_TABLES = ROOT / 'outputs' / 'tables'
OUT_FIG = ROOT / 'outputs' / 'figures'
OUT_FIG.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('OUT_METRICS =', OUT_METRICS)
print('OUT_FIG =', OUT_FIG)


## 1) Feature Group Map (group → specific columns)

In [ ]:
from src.features import feature_group_map

fg = feature_group_map()
feature_map_df = (pd.DataFrame([
    {'Feature Group': k, 'Included Feature Columns': ', '.join(v)}
    for k, v in fg.items()
])
.sort_values('Feature Group')
.reset_index(drop=True))
feature_map_df

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.axis('off')
tbl = ax.table(
    cellText=feature_map_df.values,
    colLabels=feature_map_df.columns,
    cellLoc='left',
    colLoc='left',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.35)
ax.set_title('Feature Group Map: Group → Specific Feature Columns', fontsize=14, pad=14)
plt.tight_layout()
out_path = OUT_FIG / 'poster_feature_group_map_full.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## 2) Feature Set Map (F1–F5 → specific columns)

In [ ]:
from src.config import AppConfig

config = AppConfig()
order = ['F1_momentum','F2_momentum_reversal','F3_plus_risk_liquidity','F4_plus_cross_sectional','F5_full_finance']

def expand_groups_to_cols(groups: list[str]) -> list[str]:
    cols = []
    for g in groups:
        cols.extend(fg.get(g, []))
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

rows = []
for fs in order:
    groups = config.feature_sets[fs]
    cols = expand_groups_to_cols(groups)
    rows.append({
        'Feature Set': fs,
        'Groups': ', '.join(groups),
        'Specific Feature Columns': ', '.join(cols),
        'n_cols': len(cols),
    })
feature_set_cols_df = pd.DataFrame(rows)
feature_set_cols_df

In [ ]:
show_df = feature_set_cols_df[['Feature Set','Specific Feature Columns','n_cols']].copy()
fig, ax = plt.subplots(figsize=(18, 8))
ax.axis('off')
tbl = ax.table(
    cellText=show_df.values,
    colLabels=show_df.columns,
    cellLoc='left',
    colLoc='left',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.4)
ax.set_title('Feature Sets Expanded: F1–F5 → Specific Feature Columns', fontsize=14, pad=14)
plt.tight_layout()
out_path = OUT_FIG / 'poster_feature_sets_specific_columns.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## 3) Sharpe bar chart by feature set (bars = model|horizon; highlight Logistic|3D)

In [ ]:
backtest_path = OUT_METRICS / '02_live_backtest_summary.csv'
assert backtest_path.exists(), f'Missing: {backtest_path} (run pipeline first)'
backtest = pd.read_csv(backtest_path)

bt = backtest.copy()
bt['combo'] = bt.apply(lambda r: f"{r['model']}|{int(r['horizon_days'])}D", axis=1)
pivot = bt.pivot_table(index='feature_set', columns='combo', values='sharpe', aggfunc='mean').reindex(order)
combo_order = ['logistic_regression|1D','logistic_regression|3D','random_forest|1D','random_forest|3D']
pivot = pivot.reindex(columns=[c for c in combo_order if c in pivot.columns])
pivot

In [ ]:
x = np.arange(len(pivot.index))
w = 0.18

plt.figure(figsize=(12, 5))
for i, col in enumerate(pivot.columns):
    vals = pivot[col].values
    color = '#2563eb' if col == 'logistic_regression|3D' else '#9ca3af'
    plt.bar(x + (i - (len(pivot.columns)-1)/2)*w, vals, width=w, label=col, color=color)

plt.xticks(x, pivot.index, rotation=20, ha='right')
plt.ylabel('Sharpe')
plt.title('Sharpe by Feature Set (bars = Model | Horizon; blue = Logistic | 3D)')
plt.legend(fontsize=8)
plt.tight_layout()
out_path = OUT_FIG / 'poster_sharpe_barchart_by_feature_set.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## 4) Best-performance chart (equity curve image)
If you don’t see it, it usually means the pipeline run didn’t generate it (or outputs folder path mismatch).

In [ ]:
from PIL import Image

best_path = OUT_FIG / '02_live_F5_full_finance_logistic_regression_3d_equity.png'
print('Looking for:', best_path)

if best_path.exists():
    img = Image.open(best_path)
    plt.figure(figsize=(10, 4.5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Best Strategy Equity Curve: F5 + Logistic + 3D', fontsize=14)
    plt.show()
else:
    print('Not found. Run pipeline again and ensure you are in fyp_finance_ml_v2 root.')
    # show what is available
    matches = sorted(OUT_FIG.glob('*F5_full_finance*logistic_regression*3d*equity*.png'))
    print('Matches found:', len(matches))
    for m in matches[:20]:
        print(' -', m.name)

## What to upload into Canva
- `outputs/figures/poster_feature_group_map_full.png`
- `outputs/figures/poster_feature_sets_specific_columns.png`
- `outputs/figures/02_live_rankic_heatmap.png` (pipeline)
- `outputs/figures/02_live_F5_full_finance_logistic_regression_3d_equity.png` (pipeline)
- `outputs/figures/poster_sharpe_barchart_by_feature_set.png`
